# 7 — Sequential-slide registration

Serial sections share no cells but share architecture: the same islets, ducts and vessels in
almost the same places. This notebook extracts those structures and aligns the two sections.

Two tracks run and the better one wins by QC:

* **point set** — RANSAC over islet correspondences, then ICP. No images needed.
* **image** — coarse orientation search, ECC, then a bounded B-spline. DAPI↔DAPI where the
  morphology images are mounted, cell-density rasters otherwise.

Sections are mounted by hand, so rotation is arbitrary and a flip is possible; both are
searched explicitly.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from phenocycler import load_config

cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
assert cfg.integration_mode == 'sequential', cfg.integration_mode
print('registration pixel  :', cfg.reg_pixel_um, 'um')
print('non-rigid           :', cfg.reg_nonrigid, f'(capped at {cfg.reg_max_disp_um:.0f} um)')
print('islet DBSCAN        : eps =', cfg.islet_eps_um, 'um, min_samples =', cfg.islet_min_samples)

## S3 — structures

Both modalities run the **same** algorithm, seeded on **hormone-positive cells** rather than
the endocrine label. Label seeding merges islets through scattered generic-endocrine cells;
upstream that was the difference between 2–9 islets and 112+ per section.

The PhenoCycler seed threshold is `[lineage] hormone_min_norm`, which is already 5 — the
same value the hormone floor uses — so the two definitions are symmetric for free.

In [ ]:
from phenocycler.integration.structures import run_structures

run_structures(cfg)

## S4 — registration

In [ ]:
from phenocycler.integration.register import run_register

reg = run_register(cfg, roi='panc')
reg

### QC gates

A FAIL donor is excluded from islet matching and pseudo-cell linking, but **still**
contributes donor-level composition and niche comparison — both registration-free.

In [ ]:
from phenocycler.integration.qc import run_qc

qc = run_qc(cfg, roi='panc')

### Look at the overlays

Numbers say whether the gates passed; the overlay says *how* it failed when it did. Magenta
is PhenoCycler, green is Xenium; panel 2 is after registration.

In [ ]:
from phenocycler.integration.figures import registration_overlay
from IPython.display import Image, display

for donor in reg['donor_id'] if len(reg) else []:
    p = registration_overlay(cfg, donor, 'panc')
    if p:
        print(donor)
        display(Image(filename=str(p)))